In [1]:
import pandas as pd

train_df = pd.read_csv(
    r"D:\\My_Project\\Analyse\\mobile-review-sentiment-analysis\\data\\processed\\train.csv"
)

validation_df = pd.read_csv(
    r"D:\\My_Project\\Analyse\\mobile-review-sentiment-analysis\\data\\processed\\validation.csv"
)

test_df = pd.read_csv(
    r"D:\\My_Project\\Analyse\\mobile-review-sentiment-analysis\\data\\processed\\test.csv"
)

In [2]:
label2id = {
    "Negative": 0,
    "Neutral": 1,
    "Positive": 2
}

id2label = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

train_df["label"] = train_df["sentiment"].map(label2id)

validation_df["label"] = validation_df["sentiment"].map(label2id)

test_df["label"] = test_df["sentiment"].map(label2id)

In [3]:
label2id = {
    "Negative": 0,
    "Neutral": 1,
    "Positive": 2
}

id2label = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

train_df["label"] = train_df["sentiment"].map(label2id)

validation_df["label"] = validation_df["sentiment"].map(label2id)

test_df["label"] = test_df["sentiment"].map(label2id)

In [4]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df[["clean_text", "label"]],
    preserve_index=False
)

validation_dataset = Dataset.from_pandas(
    validation_df[["clean_text", "label"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["clean_text", "label"]],
    preserve_index=False
)

c:\Users\USER\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from transformers import AutoTokenizer

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully!")


Tokenizer loaded successfully!


In [6]:
def tokenize_function(examples):

    return tokenizer(
        examples["clean_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [7]:
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_validation = validation_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

Map: 100%|██████████| 1999/1999 [00:00<00:00, 14879.38 examples/s]


In [8]:
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_validation = validation_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

Map: 100%|██████████| 1999/1999 [00:00<00:00, 14821.26 examples/s]


In [9]:
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)


def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="../models/bert_sentiment",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

print("Training arguments created successfully!")

Training arguments created successfully!


In [11]:
from transformers import AutoModelForSequenceClassification

bert_model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

print("BERT model loaded!")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4738.63it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

BERT model loaded!


In [12]:
label_map = {
    "Negative": 0,
    "Neutral": 1,
    "Positive": 2
}

In [13]:
from transformers import Trainer

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    compute_metrics=compute_metrics
)

print("Trainer created successfully!")

Trainer created successfully!


In [14]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = "D:\\My_Project\\Analyse\\mobile-review-sentiment-analysis\\models\\bert_sentiment"

tokenizer = AutoTokenizer.from_pretrained(model_path)
bert_model = AutoModelForSequenceClassification.from_pretrained(model_path)

print("Trained model loaded successfully!")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4539.07it/s]

Trained model loaded successfully!


In [15]:
validation_results = trainer.evaluate(
    tokenized_validation
)

print(validation_results)

c:\Users\USER\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
No log,1.104867,0,0.323323,0.494249,0.323323,0.359200


{'eval_loss': 1.1048674583435059, 'eval_accuracy': 0.3233233233233233, 'eval_precision': 0.4942488921581193, 'eval_recall': 0.3233233233233233, 'eval_f1': 0.3591997158273032}


In [16]:
test_results = trainer.evaluate(
    tokenized_test
)

print(test_results)

c:\Users\USER\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
No log,1.102625,0,0.325663,0.485282,0.325663,0.359716


{'eval_loss': 1.1026246547698975, 'eval_accuracy': 0.32566283141570784, 'eval_precision': 0.4852815935226715, 'eval_recall': 0.32566283141570784, 'eval_f1': 0.35971621681551985}
